# 01 · SED Data Pull
Pull Social and Economic Determinants (SED) survey responses from the
**All of Us** Controlled Tier Dataset v7 via BigQuery.

| Variable | Concept ID |
|----------|------------|
| Education (Highest Grade) | 1585940 |
| Employment Status | 1585952 |
| Annual Income | 1585375 |
| Health Insurance | 1585386 |
| Home Ownership | 1585370 |


---
### Citation

This pipeline implements the iSDI construction described in:

> **Reference:** Gupta S, Lam V, Jordan IK, Mariño-Ramírez L. *A composite socioeconomic deprivation index from All of Us survey data: associations with health outcomes and disparities.* medRxiv 2024.10.04.24314904. PMID: [39802760](https://pubmed.ncbi.nlm.nih.gov/39802760/). doi: [10.1101/2024.10.04.24314904](https://doi.org/10.1101/2024.10.04.24314904)


In [ ]:
"""
01_sed_data_pull.py
Pull SED survey responses from AoU CDR v7.
"""

import os
import numpy as np
import pandas as pd


## Helper: pull one survey concept

In [ ]:
def pull_survey_concept(concept_id: int, label: str) -> pd.DataFrame:
    """Query ds_survey for a single question concept."""
    sql = f"""
        SELECT
            answer.person_id,
            answer.question,
            answer.answer
        FROM
            `{os.environ["WORKSPACE_CDR"]}.ds_survey` answer
        WHERE
            question_concept_id IN ({concept_id})
    """
    df = pd.read_gbq(
        sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook",
    )
    print(f"[{label}] concept_id={concept_id}  rows={len(df):,}")
    return df


## Pull each SED domain

In [ ]:
edu_df = pull_survey_concept(1585940, "Education")   # Highest Grade
emp_df = pull_survey_concept(1585952, "Employment")  # Employment Status
inc_df = pull_survey_concept(1585375, "Income")      # Annual Income
ins_df = pull_survey_concept(1585386, "Insurance")   # Health Insurance
own_df = pull_survey_concept(1585370, "Housing")     # Current Home Own
